In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

shutil.copy('/content/drive/MyDrive/Capstone2/xgboost_model_v1.pkl', 'xgboost_model_v1.pkl')
shutil.copy('/content/drive/MyDrive/Capstone2/pipeline_vars_v1.pkl', 'pipeline_vars_v1.pkl')
print("✅ Files loaded")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Files loaded


In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import pickle
import shap
import matplotlib.pyplot as plt

@st.cache_resource
def load_model():
    model = joblib.load("xgboost_model_v1.pkl")
    with open("pipeline_vars_v1.pkl", "rb") as f:
        data = pickle.load(f)
    return model, data

model, data = load_model()
feature_names = data["feature_names"]
le_dict       = data["le_dict"]
THRESHOLD     = 0.40

st.set_page_config(page_title="Vision Zero Simulator", page_icon="🚦", layout="wide")
st.title("🚦 Vision Zero Injury Risk Simulator")
st.markdown("**Montgomery County Road Safety** — XGBoost | 91% Injury Recall")
st.markdown("---")

col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("**Environmental**")
    weather         = st.selectbox("Weather", ["CLEAR","CLOUDY","RAINING","SNOW","FOGGY","UNKNOWN"])
    surface         = st.selectbox("Surface Condition", ["DRY","WET","ICE","SNOW","UNKNOWN"])
    light           = st.selectbox("Light Condition", ["DAYLIGHT","DARK LIGHTS ON","DARK NO LIGHTS","DAWN","DUSK","UNKNOWN"])
    traffic_control = st.selectbox("Traffic Control", ["NO CONTROLS","TRAFFIC SIGNAL","STOP SIGN","YIELD SIGN","UNKNOWN"])

with col2:
    st.markdown("**Crash Details**")
    collision_type   = st.selectbox("Collision Type", ["REAR END","HEAD ON","ANGLE","SIDESWIPE SAME DIRECTION","SINGLE VEHICLE","OTHER"])
    vehicle_movement = st.selectbox("Vehicle Movement", ["GOING STRAIGHT","TURNING LEFT","TURNING RIGHT","CHANGING LANES","STOPPED IN TRAFFIC","OTHER"])
    speed_limit      = st.slider("Speed Limit (mph)", 5, 75, 35, step=5)
    driver_at_fault  = st.selectbox("Driver At Fault", ["Yes","No"])
    vehicle_damage   = st.selectbox("Vehicle Damage Extent", ["SUPERFICIAL","FUNCTIONAL","DISABLING","DESTROYED","NO DAMAGE","UNKNOWN"])

with col3:
    st.markdown("**Driver & Vehicle**")
    substance_abuse = st.selectbox("Driver Substance Abuse", ["NONE DETECTED","ALCOHOL PRESENT","ALCOHOL CONTRIBUTED","UNKNOWN"])
    vehicle_body    = st.selectbox("Vehicle Body Type", ["PASSENGER CAR","PICKUP TRUCK","SUV","MOTORCYCLE","BUS","OTHER"])
    vehicle_year    = st.slider("Vehicle Year", 1990, 2024, 2015)
    crash_hour      = st.slider("Hour of Crash (0-23)", 0, 23, 12)
    crash_day       = st.selectbox("Day of Week", ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])

st.markdown("---")

if st.button("🔍 Predict Injury Risk", type="primary"):
    day_map    = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
    dayofweek  = day_map[crash_day]
    is_night   = 1 if (crash_hour >= 22 or crash_hour <= 5) else 0
    is_weekend = 1 if dayofweek >= 5 else 0
    vehicle_age = 2024 - vehicle_year

    def encode_val(col, val, le_dict):
        mapping  = le_dict.get(col, {})
        val_up   = str(val).upper().strip()
        if val_up in mapping: return mapping[val_up]
        if val in mapping:    return mapping[val]
        return 0

    raw = {
        "Weather": weather, "Surface Condition": surface,
        "Light": light, "Traffic Control": traffic_control,
        "Collision Type": collision_type, "Circumstance": "UNKNOWN",
        "Vehicle Movement": vehicle_movement, "Speed Limit": speed_limit,
        "Driver Substance Abuse": substance_abuse,
        "Driver At Fault": "YES" if driver_at_fault == "Yes" else "NO",
        "Vehicle Damage Extent": vehicle_damage, "Vehicle Body Type": vehicle_body,
        "Route Type": "UNKNOWN", "Vehicle Going Dir": "UNKNOWN",
        "Driverless Vehicle": "No", "Parked Vehicle": "No",
        "Vehicle First Impact Location": "UNKNOWN",
        "Driver Distracted By": "NOT DISTRACTED",
        "Non-Motorist Substance Abuse": "NONE DETECTED",
        "Latitude": 39.1, "Longitude": -77.2,
        "Hour": crash_hour, "DayOfWeek": dayofweek,
        "IsNight": is_night, "IsWeekend": is_weekend,
        "Vehicle Age": vehicle_age,
    }

    encoded = {}
    for col, val in raw.items():
        encoded[col] = encode_val(col, val, le_dict) if col in le_dict else val

    input_df = pd.DataFrame([encoded])
    input_df["Speed_x_Night"]          = input_df["Speed Limit"] * input_df["IsNight"]
    input_df["Substance_x_Night"]      = input_df["Driver Substance Abuse"] * input_df["IsNight"]
    input_df["VehicleAge_x_Collision"] = input_df["Vehicle Age"] * input_df["Collision Type"]
    input_df = input_df[feature_names]

    proba     = model.predict_proba(input_df)[0][1]
    predicted = 1 if proba >= THRESHOLD else 0

    r1, r2, r3 = st.columns(3)
    with r1:
        if predicted == 1:
            st.error("⚠️ INJURY PREDICTED")
        else:
            st.success("✅ NO INJURY PREDICTED")
    with r2:
        st.metric("Injury Probability", f"{proba:.1%}")
    with r3:
        if proba >= 0.70:   risk = "🔴 HIGH RISK"
        elif proba >= 0.40: risk = "🟡 MODERATE RISK"
        else:               risk = "🟢 LOW RISK"
        st.metric("Risk Level", risk)

    st.progress(float(proba))
    st.markdown("---")

    if predicted == 1:
        st.warning(f"This scenario is flagged as HIGH RISK with {proba:.1%} injury probability. Consider speed limit reduction, improved lighting, or additional traffic controls.")
    else:
        st.info(f"This scenario has a {proba:.1%} injury probability. Lower risk based on historical crash patterns.")

    st.subheader("Why did the model predict this? (SHAP)")
    try:
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(input_df)
        shap_vals   = shap_values[1] if isinstance(shap_values, list) else shap_values
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.plots.waterfall(shap.Explanation(
            values      = shap_vals[0],
            base_values = explainer.expected_value[1] if isinstance(
                explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
            data          = input_df.iloc[0],
            feature_names = feature_names
        ), show=False)
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        st.caption("Blue = pushes toward No Injury. Pink/Red = pushes toward Injury.")
    except Exception as e:
        st.warning(f"SHAP unavailable: {e}")

st.markdown("---")
st.markdown("**Model:** XGBoost | Threshold=0.40 | 91% Recall | **Built by:** Josh Bui & Danson Vo | Capstone 2")
'''

with open("app.py", "w") as f:
    f.write(app_code)
print("✅ app.py created")

✅ app.py created


In [ ]:
!pip install pyngrok streamlit

In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token("3C0Lq5aIbY5zQsM4sgGw3cVgZMh_3CPczDzSaNAzeovAbtbeA")

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                   '--server.port=8501',
                   '--server.headless=true',
                   '--server.enableCORS=false'])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

print("Waiting 20 seconds for Streamlit...")
time.sleep(20)

public_url = ngrok.connect(8501)
print(f"\n✅ Live at: {public_url}")

Waiting 20 seconds for Streamlit...

✅ Live at: NgrokTunnel: "https://unrecited-naovely-creola.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# ═══════════════════════════════════════════════════════════════
# MONTGOMERY COUNTY ROAD SAFETY PREDICTION
# Data Science Capstone 2 — V1 FINAL (High Recall)
# XGBoost + SMOTETomek + SHAP + Fairness
# Target: 91% Injury Recall at Threshold 0.40
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             recall_score, precision_score, accuracy_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
import shap
import joblib
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ── 1. LOAD & CLEAN ──────────────────────────────────────────────────────────
df = pd.read_csv('Crash_Reporting_-_Drivers_Data.csv', low_memory=False)

df['Injury Severity'] = df['Injury Severity'].str.upper().str.strip()

binary_map = {
    'NO APPARENT INJURY': 0,
    'POSSIBLE INJURY': 1,
    'SUSPECTED MINOR INJURY': 1,
    'SUSPECTED SERIOUS INJURY': 1,
    'FATAL INJURY': 1
}
df['Severity_Class'] = df['Injury Severity'].map(binary_map)
df = df.dropna(subset=['Severity_Class'])
df['Severity_Class'] = df['Severity_Class'].astype(int)

print("Class distribution:")
print(df['Severity_Class'].value_counts())
print(df['Severity_Class'].value_counts(normalize=True).round(3))

# ── 2. FEATURE ENGINEERING ───────────────────────────────────────────────────
df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'], errors='coerce')
df['Hour']        = df['Crash Date/Time'].dt.hour
df['DayOfWeek']   = df['Crash Date/Time'].dt.dayofweek
df['IsNight']     = ((df['Hour'] >= 22) | (df['Hour'] <= 5)).astype(int)
df['IsWeekend']   = (df['DayOfWeek'] >= 5).astype(int)
df['Vehicle Age'] = 2024 - df['Vehicle Year'].replace(0, np.nan)

features = [
    'Weather', 'Surface Condition', 'Light', 'Traffic Control',
    'Collision Type', 'Circumstance', 'Vehicle Movement',
    'Speed Limit', 'Driver Substance Abuse', 'Driver At Fault',
    'Vehicle Damage Extent', 'Vehicle Body Type', 'Route Type',
    'Vehicle Going Dir', 'Driverless Vehicle', 'Parked Vehicle',
    'Vehicle First Impact Location', 'Driver Distracted By',
    'Non-Motorist Substance Abuse', 'Latitude', 'Longitude',
    'Hour', 'DayOfWeek', 'IsNight', 'IsWeekend', 'Vehicle Age'
]

# ── 3. PREP DATA ─────────────────────────────────────────────────────────────
df_model = df[features + ['Severity_Class']].copy()

for col in features:
    if df_model[col].dtype == 'object':
        df_model[col] = df_model[col].fillna('UNKNOWN')
    else:
        df_model[col] = df_model[col].fillna(df_model[col].median())

le = LabelEncoder()
le_dict = {}
for col in features:
    if df_model[col].dtype == 'object':
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        le_dict[col] = dict(zip(le.classes_, le.transform(le.classes_)))

X = df_model[features]
y = df_model['Severity_Class']

print(f"\nDataset shape: {df_model.shape}")

# ── 4. FEATURE INTERACTIONS ──────────────────────────────────────────────────
def add_interactions(X):
    X = X.copy()
    X['Speed_x_Night']          = X['Speed Limit']            * X['IsNight']
    X['Substance_x_Night']      = X['Driver Substance Abuse'] * X['IsNight']
    X['VehicleAge_x_Collision'] = X['Vehicle Age']            * X['Collision Type']
    return X

# ── 5. TRAIN/TEST SPLIT ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = add_interactions(X_train)
X_test  = add_interactions(X_test)
feature_names = list(X_train.columns)
print(f"\nTotal features: {len(feature_names)}")

# ── 6. SMOTE TOMEK (train only) ──────────────────────────────────────────────
smt = SMOTETomek(random_state=42)
X_train_sm, y_train_sm = smt.fit_resample(X_train, y_train)
print("After SMOTETomek:", pd.Series(y_train_sm).value_counts())

# ── 7. TRAIN ALL MODELS ──────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, class_weight='balanced', max_depth=20, random_state=42),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        class_weight='balanced', random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=5, use_label_encoder=False,
        eval_metric='logloss', random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    rec    = recall_score(y_test, y_pred)
    prec   = precision_score(y_test, y_pred, zero_division=0)
    results[name] = {'model': model, 'y_pred': y_pred,
                     'accuracy': acc, 'recall': rec, 'precision': prec}
    print(f"\n── {name} ──")
    print(f"Accuracy: {acc:.3f} | Recall: {rec:.3f} | Precision: {prec:.3f}")
    print(classification_report(y_test, y_pred,
                                target_names=['No Injury', 'Injury']))

print("\n── All Model Results ──")
for name, r in results.items():
    print(f"{name}: Acc={r['accuracy']:.3f} | Recall={r['recall']:.3f} | Precision={r['precision']:.3f}")

# ── 8. THRESHOLD TUNING ON XGBOOST ──────────────────────────────────────────
best_model   = results['XGBoost']['model']
y_proba      = best_model.predict_proba(X_test)
injury_proba = y_proba[:, 1]

print("\nThreshold | Accuracy | Recall | Precision")
print("-" * 50)
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred_t = (injury_proba >= t).astype(int)
    acc      = accuracy_score(y_test, y_pred_t)
    rec      = recall_score(y_test, y_pred_t)
    prec     = precision_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:.2f}      | {acc:.3f}    | {rec:.3f}  | {prec:.3f}")

# ── 9. FINAL MODEL ───────────────────────────────────────────────────────────
THRESHOLD    = 0.40
y_pred_final = (injury_proba >= THRESHOLD).astype(int)

print(f"\n── Final Model: XGBoost V1 | Threshold={THRESHOLD} ──")
print(classification_report(y_test, y_pred_final,
                            target_names=['No Injury', 'Injury']))

# ── 10. CONFUSION MATRIX ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Injury', 'Injury'],
            yticklabels=['No Injury', 'Injury'], ax=ax)
ax.set_title(f'Confusion Matrix — XGBoost V1 (threshold={THRESHOLD})')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved confusion_matrix.png")

# ── 11. SHAP ─────────────────────────────────────────────────────────────────
print("\nCalculating SHAP values...")
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
shap_vals   = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_test, feature_names=feature_names,
                  plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved shap_bar.png")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_test, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved shap_beeswarm.png")

# ── 12. SHAP LOCAL ───────────────────────────────────────────────────────────
false_negatives = X_test[(y_test == 1) & (y_pred_final == 0)]
print(f"\nFalse Negatives: {len(false_negatives)}")
if len(false_negatives) > 0:
    fn_idx   = false_negatives.index[0]
    loc      = X_test.index.get_loc(fn_idx)
    base_val = explainer.expected_value[1] if isinstance(
        explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.plots.waterfall(shap.Explanation(
        values        = shap_vals[loc],
        base_values   = base_val,
        data          = X_test.iloc[loc],
        feature_names = feature_names
    ), show=False)
    plt.tight_layout()
    plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved shap_waterfall.png")

# ── 13. FAIRNESS ─────────────────────────────────────────────────────────────
test_indices = X_test.index
df_test_orig = df.loc[test_indices].copy()
df_test_orig['y_true']       = y_test.values
df_test_orig['y_pred']       = y_pred_final
df_test_orig['injury_proba'] = injury_proba

def group_metrics(df, group_col, min_samples=100):
    results = []
    for group, subset in df.groupby(group_col):
        if len(subset) < min_samples or subset['y_true'].nunique() < 2:
            continue
        results.append({
            'Group':     str(group),
            'N':         len(subset),
            'Injuries':  int(subset['y_true'].sum()),
            'Recall':    round(recall_score(subset['y_true'], subset['y_pred'], zero_division=0), 3),
            'Precision': round(precision_score(subset['y_true'], subset['y_pred'], zero_division=0), 3),
            'Accuracy':  round(accuracy_score(subset['y_true'], subset['y_pred']), 3),
        })
    return pd.DataFrame(results).sort_values('Recall')

print("\n── Fairness: Light Condition ──")
light_df = group_metrics(df_test_orig, 'Light')
print(light_df.to_string(index=False))

print("\n── Fairness: Weather ──")
weather_df = group_metrics(df_test_orig, 'Weather')
print(weather_df.to_string(index=False))

if 'Municipality' in df_test_orig.columns:
    print("\n── Fairness: Municipality ──")
    muni_df = group_metrics(df_test_orig, 'Municipality')
    print(muni_df.to_string(index=False))

df_test_orig['Time_of_Day'] = df_test_orig['IsNight'].map({1:'Night', 0:'Day'})
df_test_orig['Day_Type']    = df_test_orig['IsWeekend'].map({1:'Weekend', 0:'Weekday'})

print("\n── Fairness: Day vs Night ──")
print(group_metrics(df_test_orig, 'Time_of_Day', min_samples=50).to_string(index=False))

print("\n── Fairness: Weekday vs Weekend ──")
print(group_metrics(df_test_orig, 'Day_Type', min_samples=50).to_string(index=False))

if 'Municipality' in df_test_orig.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    muni_sorted = muni_df.sort_values('Recall')
    colors = ['#EF4444' if r < 0.7 else '#F59E0B' if r < 0.85 else '#22C55E'
              for r in muni_sorted['Recall']]
    ax.barh(muni_sorted['Group'], muni_sorted['Recall'], color=colors)
    ax.axvline(x=muni_df['Recall'].mean(), color='navy', linestyle='--',
               linewidth=2, label=f"Mean: {muni_df['Recall'].mean():.2f}")
    ax.set_xlabel('Injury Recall')
    ax.set_title('Fairness: Injury Recall by Municipality')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fairness_municipality.png', dpi=150, bbox_inches='tight')
    plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
light_sorted = light_df.sort_values('Recall')
colors = ['#EF4444' if r < 0.7 else '#F59E0B' if r < 0.85 else '#22C55E'
          for r in light_sorted['Recall']]
ax.barh(light_sorted['Group'], light_sorted['Recall'], color=colors)
ax.axvline(x=light_df['Recall'].mean(), color='navy', linestyle='--',
           linewidth=2, label=f"Mean: {light_df['Recall'].mean():.2f}")
ax.set_xlabel('Injury Recall')
ax.set_title('Fairness: Injury Recall by Light Condition')
ax.legend()
plt.tight_layout()
plt.savefig('fairness_light.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 14. SAVE TO GOOGLE DRIVE ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/Capstone2', exist_ok=True)

joblib.dump(best_model, '/content/drive/MyDrive/Capstone2/xgboost_model_v1.pkl')

with open('/content/drive/MyDrive/Capstone2/pipeline_vars_v1.pkl', 'wb') as f:
    pickle.dump({
        'feature_names': feature_names,
        'le_dict':       le_dict,
        'X_test':        X_test,
        'y_test':        y_test,
        'y_pred_final':  y_pred_final,
        'shap_vals':     shap_vals,
        'injury_proba':  injury_proba,
        'X_train_sm':    X_train_sm,
        'y_train_sm':    y_train_sm,
        'THRESHOLD':     THRESHOLD,
    }, f)

print("\n✅ V1 Pipeline complete. Everything saved to Google Drive.")
print(f"Model: XGBoost V1 | Threshold={THRESHOLD}")
print(f"Features: {len(feature_names)}")

Class distribution:
Severity_Class
0    156017
1     34023
Name: count, dtype: int64
Severity_Class
0    0.821
1    0.179
Name: proportion, dtype: float64

Dataset shape: (190040, 27)

Total features: 29
After SMOTETomek: Severity_Class
0    124071
1    124071
Name: count, dtype: int64

── Logistic Regression ──
Accuracy: 0.669 | Recall: 0.599 | Precision: 0.293
              precision    recall  f1-score   support

   No Injury       0.89      0.68      0.77     31203
      Injury       0.29      0.60      0.39      6805

    accuracy                           0.67     38008
   macro avg       0.59      0.64      0.58     38008
weighted avg       0.78      0.67      0.70     38008


── Random Forest ──
Accuracy: 0.780 | Recall: 0.387 | Precision: 0.386
              precision    recall  f1-score   support

   No Injury       0.87      0.87      0.87     31203
      Injury       0.39      0.39      0.39      6805

    accuracy                           0.78     38008
   macro avg      

ValueError: mount failed

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import pickle
import shap
import matplotlib.pyplot as plt

# ── LOAD MODEL & VARS ────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    model = joblib.load("xgboost_model_v1.pkl")
    with open("pipeline_vars_v1.pkl", "rb") as f:
        data = pickle.load(f)
    return model, data

model, data = load_model()
feature_names = data["feature_names"]
le_dict       = data["le_dict"]
THRESHOLD     = 0.40

# ── PAGE CONFIG ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Vision Zero Simulator",
    page_icon="🚦",
    layout="wide"
)

# ── HEADER ───────────────────────────────────────────────────────────────────
st.title("🚦 Vision Zero Injury Risk Simulator")
st.markdown("**Montgomery County Road Safety Prediction** — XGBoost Model | 91% Injury Recall")
st.markdown("---")

st.markdown("""
This tool helps transportation planners predict the probability of injury in a crash scenario.
Input road and environmental conditions below to get a real-time injury risk score.
""")

# ── INPUT FORM ───────────────────────────────────────────────────────────────
st.subheader("Enter Crash Conditions")

col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("**Environmental**")
    weather = st.selectbox("Weather", [
        "CLEAR", "CLOUDY", "RAINING", "SNOW", "FOGGY", "UNKNOWN"])
    surface = st.selectbox("Surface Condition", [
        "DRY", "WET", "ICE", "SNOW", "SAND/MUD/DIRT/OIL/GRAVEL", "UNKNOWN"])
    light = st.selectbox("Light Condition", [
        "DAYLIGHT", "DARK LIGHTS ON", "DARK NO LIGHTS", "DAWN", "DUSK", "UNKNOWN"])
    traffic_control = st.selectbox("Traffic Control", [
        "NO CONTROLS", "TRAFFIC SIGNAL", "STOP SIGN", "YIELD SIGN",
        "FLASHING TRAFFIC SIGNAL", "UNKNOWN"])

with col2:
    st.markdown("**Crash Details**")
    collision_type = st.selectbox("Collision Type", [
        "REAR END", "HEAD ON", "ANGLE", "SIDESWIPE SAME DIRECTION",
        "SIDESWIPE OPPOSITE DIRECTION", "SINGLE VEHICLE", "OTHER"])
    vehicle_movement = st.selectbox("Vehicle Movement", [
        "GOING STRAIGHT", "TURNING LEFT", "TURNING RIGHT",
        "CHANGING LANES", "PARKED", "STOPPED IN TRAFFIC", "OTHER"])
    speed_limit = st.slider("Speed Limit (mph)", 5, 75, 35, step=5)
    driver_at_fault = st.selectbox("Driver At Fault", ["Yes", "No"])
    vehicle_damage  = st.selectbox("Vehicle Damage Extent", [
        "SUPERFICIAL", "FUNCTIONAL", "DISABLING", "DESTROYED", "NO DAMAGE", "UNKNOWN"])

with col3:
    st.markdown("**Driver & Vehicle**")
    substance_abuse = st.selectbox("Driver Substance Abuse", [
        "NONE DETECTED", "ALCOHOL PRESENT", "ALCOHOL CONTRIBUTED",
        "ILLEGAL DRUG CONTRIBUTED", "UNKNOWN"])
    vehicle_body = st.selectbox("Vehicle Body Type", [
        "PASSENGER CAR", "PICKUP TRUCK", "SUV", "MOTORCYCLE",
        "BUS", "LARGE TRUCK", "OTHER"])
    vehicle_year = st.slider("Vehicle Year", 1990, 2024, 2015)
    crash_hour   = st.slider("Hour of Crash (0-23)", 0, 23, 12)
    crash_day    = st.selectbox("Day of Week", [
        "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

st.markdown("---")

# ── PREDICT BUTTON ───────────────────────────────────────────────────────────
if st.button("🔍 Predict Injury Risk", type="primary"):

    day_map = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,
               "Friday":4,"Saturday":5,"Sunday":6}
    dayofweek  = day_map[crash_day]
    is_night   = 1 if (crash_hour >= 22 or crash_hour <= 5) else 0
    is_weekend = 1 if dayofweek >= 5 else 0
    vehicle_age = 2024 - vehicle_year

    def encode_val(col, val, le_dict):
        mapping = le_dict.get(col, {})
        val_upper = str(val).upper().strip()
        if val_upper in mapping:
            return mapping[val_upper]
        if val in mapping:
            return mapping[val]
        return 0

    raw_input = {
        "Weather":                     weather,
        "Surface Condition":           surface,
        "Light":                       light,
        "Traffic Control":             traffic_control,
        "Collision Type":              collision_type,
        "Circumstance":                "UNKNOWN",
        "Vehicle Movement":            vehicle_movement,
        "Speed Limit":                 speed_limit,
        "Driver Substance Abuse":      substance_abuse,
        "Driver At Fault":             "YES" if driver_at_fault == "Yes" else "NO",
        "Vehicle Damage Extent":       vehicle_damage,
        "Vehicle Body Type":           vehicle_body,
        "Route Type":                  "UNKNOWN",
        "Vehicle Going Dir":           "UNKNOWN",
        "Driverless Vehicle":          "No",
        "Parked Vehicle":              "No",
        "Vehicle First Impact Location":"UNKNOWN",
        "Driver Distracted By":        "NOT DISTRACTED",
        "Non-Motorist Substance Abuse":"NONE DETECTED",
        "Latitude":                    39.1,
        "Longitude":                   -77.2,
        "Hour":                        crash_hour,
        "DayOfWeek":                   dayofweek,
        "IsNight":                     is_night,
        "IsWeekend":                   is_weekend,
        "Vehicle Age":                 vehicle_age,
    }

    encoded = {}
    for col, val in raw_input.items():
        if col in le_dict:
            encoded[col] = encode_val(col, val, le_dict)
        else:
            encoded[col] = val

    input_df = pd.DataFrame([encoded])

    # Add interactions
    input_df["Speed_x_Night"]          = input_df["Speed Limit"] * input_df["IsNight"]
    input_df["Substance_x_Night"]      = input_df["Driver Substance Abuse"] * input_df["IsNight"]
    input_df["VehicleAge_x_Collision"] = input_df["Vehicle Age"] * input_df["Collision Type"]

    input_df = input_df[feature_names]

    # Predict
    proba     = model.predict_proba(input_df)[0][1]
    predicted = 1 if proba >= THRESHOLD else 0

    # ── RESULTS ──────────────────────────────────────────────────────────────
    st.subheader("Prediction Results")
    res_col1, res_col2, res_col3 = st.columns(3)

    with res_col1:
        if predicted == 1:
            st.error(f"⚠️ INJURY PREDICTED")
        else:
            st.success(f"✅ NO INJURY PREDICTED")

    with res_col2:
        st.metric("Injury Probability", f"{proba:.1%}")

    with res_col3:
        if proba >= 0.70:
            risk = "🔴 HIGH RISK"
        elif proba >= 0.40:
            risk = "🟡 MODERATE RISK"
        else:
            risk = "🟢 LOW RISK"
        st.metric("Risk Level", risk)

    # Risk bar
    st.markdown("**Injury Probability Score:**")
    st.progress(float(proba))

    # Interpretation
    st.markdown("---")
    st.subheader("How to Interpret This")

    if predicted == 1:
        st.warning(f"""
        **This scenario is flagged as HIGH RISK.**
        The model predicts a {proba:.1%} probability of injury.
        This road segment and condition combination warrants attention from safety planners.
        Consider: speed limit reduction, improved lighting, additional traffic controls.
        """)
    else:
        st.info(f"""
        **This scenario is predicted as lower risk.**
        The model predicts a {proba:.1%} probability of injury.
        Note: Our model has 91% recall — it catches most real injuries.
        A low score here suggests this combination has historically lower injury rates.
        """)

    # ── SHAP EXPLANATION ─────────────────────────────────────────────────────
    st.markdown("---")
    st.subheader("Why did the model predict this? (SHAP Explanation)")

    try:
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(input_df)
        shap_vals   = shap_values[1] if isinstance(shap_values, list) else shap_values

        fig, ax = plt.subplots(figsize=(10, 6))
        shap.plots.waterfall(shap.Explanation(
            values        = shap_vals[0],
            base_values   = explainer.expected_value[1] if isinstance(
                explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
            data          = input_df.iloc[0],
            feature_names = feature_names
        ), show=False)
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()

        st.caption("""
        Blue bars push the prediction toward NO INJURY.
        Pink/red bars push toward INJURY.
        The longer the bar, the stronger the influence.
        """)
    except Exception as e:
        st.warning(f"SHAP explanation unavailable: {e}")

# ── FOOTER ───────────────────────────────────────────────────────────────────
st.markdown("---")
st.markdown("""
**Model Info:** XGBoost | Threshold=0.40 | 91% Injury Recall | 201K Montgomery County crashes
**Limitations:** Vehicle Damage Extent is a post-crash variable. Precision is 26% — false alarms are expected.
**Built by:** Josh Bui & Danson Vo | Data Science Capstone 2
""")
'''

# Save the app file
with open("app.py", "w") as f:
    f.write(app_code)
print("✅ app.py saved")

# Save model files to current directory for the app
import shutil
shutil.copy('/content/drive/MyDrive/Capstone2/xgboost_model_v1.pkl', 'xgboost_model_v1.pkl')
shutil.copy('/content/drive/MyDrive/Capstone2/pipeline_vars_v1.pkl', 'pipeline_vars_v1.pkl')
print("✅ Model files copied")

# Install and run streamlit
import subprocess
subprocess.run(['pip', 'install', 'streamlit', 'pyngrok', '--break-system-packages', '-q'])
print("✅ Streamlit installed")

In [ ]:
from pyngrok import ngrok
import subprocess
import threading
import time

# Add your auth token here
ngrok.set_auth_token("3C0Lq5aIbY5zQsM4sgGw3cVgZMh_3CPczDzSaNAzeovAbtbeA")

# Kill any existing tunnels
ngrok.kill()

# Start streamlit in background
def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                   '--server.port=8501',
                   '--server.headless=true'])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print(f"\n✅ Your app is live at: {public_url}")
print("Open that link to see your Vision Zero Simulator")


✅ Your app is live at: NgrokTunnel: "https://unrecited-naovely-creola.ngrok-free.dev" -> "http://localhost:8501"
Open that link to see your Vision Zero Simulator
